<a href="https://colab.research.google.com/github/santannalorena936/Computa-o-25.2-/blob/main/CodigoFonte_Assistencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
streamlit_code = """
# ================= IMPORTAÇÕES =================
import streamlit as st              # Interface web
import sqlite3                     # Banco de dados SQLite
from datetime import datetime      # Datas
import os                          # Manipulação de arquivos
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas  # Geração de PDF
import pandas as pd               # Tabelas
import openpyxl                   # Exportar Excel
import io                         # Arquivo em memória

# ================= BANCO =================
# Cria conexão com banco local
conn = sqlite3.connect("ordens.db", check_same_thread=False)
cursor = conn.cursor()

# Cria tabela se não existir
cursor.execute('''
CREATE TABLE IF NOT EXISTS ordens (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    loja TEXT,
    cliente TEXT,
    equipamento TEXT,
    marca TEXT,
    modelo TEXT,
    defeito TEXT,
    data_entrada TEXT,
    data_reparo TEXT,
    mes_reparo TEXT
)
'''
)
conn.commit()

# ================= FUNÇÃO FINALIZAR OS =================
# Atualiza a OS com data atual e mês
def finalizar_os(os_id):
    data_atual = datetime.now()

    data_reparo = data_atual.strftime("%d/%m/%Y")
    mes_reparo = data_atual.strftime("%B").capitalize()

    cursor.execute('''
    UPDATE ordens
    SET data_reparo = ?, mes_reparo = ?
    WHERE id = ?
    ''', (data_reparo, mes_reparo, os_id))

    conn.commit()

# ================= FUNÇÃO PDF =================
def gerar_pdf(os_data):

    # Cria pasta se não existir
    if not os.path.exists("pdf"):
        os.makedirs("pdf")

    nome = f"pdf/OS_{os_data[0]}.pdf"
    c = canvas.Canvas(nome, pagesize=A4)

    c.setFont("Helvetica", 12)

    # Dados principais
    c.drawString(200, 800, f"ORDEM DE SERVIÇO Nº {os_data[0]}")
    c.drawString(50, 750, f"Loja: {os_data[1]}")
    c.drawString(50, 730, f"Cliente: {os_data[2]}")
    c.drawString(50, 710, f"Equipamento: {os_data[3]}")
    c.drawString(50, 690, f"Marca: {os_data[4]}")
    c.drawString(50, 670, f"Modelo: {os_data[5]}")
    c.drawString(50, 650, f"Defeito: {os_data[6]}")

    # Datas
    c.drawString(50, 630, f"Data entrada: {os_data[7]}")
    c.drawString(50, 610, f"Data reparo: {os_data[8] if os_data[8] else 'Em andamento'}")
    c.drawString(50, 590, f"Mês reparo: {os_data[9] if os_data[9] else '-'}")

    c.save()
    return nome

# ================= INTERFACE =================
st.title("Sistema de Gestão de Manutenção – Belessa")

# Lista de lojas
LOJAS = [
    "Belessa Alagoinhas","Belessa Aracaju Centro","Belessa Aracaju Jardins",
    "Belessa Arapiraca","Belessa Av. Sete","Belessa Barreiras",
    "Belessa Boca do Rio","Belessa Cajazeiras","Belessa Camaçari",
    "Belessa Candeias","Belessa Cruz das Almas","Belessa Dias D'Ávila",
    "Belessa Eunapolis","Belessa Feira de Santana","Belessa Ilheus",
    "Belessa Itabaiana","Belessa Itabuna","Belessa Itaigara",
    "Belessa Itapuã","Belessa Jequie","Belessa Juazeiro",
    "Belessa Lagarto","Belessa Lauro de Freitas","Belessa Maceió Centro",
    "Belessa Maceió Jatiuca","Belessa Pau da Lima","Belessa Paulo Afonso",
    "Belessa Periperi","Belessa Porto Seguro","Belessa SAJ",
    "Belessa Simões Filho","Belessa Socorro","Belessa Uruguai",
    "Belessa Valença","Belessa Vitoria da Conquista","Loja não identificada"
]

# Lista de equipamentos
EQUIPAMENTOS = ["Motor porquinho", "Cabine", "Coletor", "Luminária", "Outros"]

# ================= CADASTRO =================
with st.form("form_os"):
    st.header("Cadastro de Ordem de Serviço")

    loja = st.selectbox("Loja", LOJAS)
    cliente = st.text_input("Cliente")
    equipamento = st.selectbox("Equipamento", EQUIPAMENTOS)
    marca = st.text_input("Marca")
    modelo = st.text_input("Modelo")
    defeito = st.text_area("Defeito")

    data_entrada = st.date_input("Data de entrada")

    submitted = st.form_submit_button("Cadastrar OS")

    if submitted:
        if not cliente:
            st.error("Digite o cliente!")
        else:
            # Dados iniciais (OS começa pendente)
            dados = (
                loja,
                cliente,
                equipamento,
                marca,
                modelo,
                defeito,
                data_entrada.strftime("%d/%m/%Y"),
                None,
                ""
            )

            # Inserção no banco
            cursor.execute('''
                INSERT INTO ordens
                (loja, cliente, equipamento, marca, modelo, defeito, data_entrada, data_reparo, mes_reparo)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', dados)

            conn.commit()

            os_id = cursor.lastrowid

            # Busca OS completa para gerar PDF
            cursor.execute("SELECT * FROM ordens WHERE id = ?", (os_id,))
            new_os_data = cursor.fetchone()

            pdf = gerar_pdf(new_os_data)

            st.success(f"OS Nº {os_id} criada!")

            # Download do PDF
            with open(pdf, "rb") as f:
                st.download_button("📄 Baixar PDF", f, file_name=os.path.basename(pdf))

# ================= LISTAGEM =================
st.header("📊 Lista de Ordens de Serviço")

# Campo de busca
termo_busca = st.text_input("🔍 Buscar por cliente")

# Consulta no banco
if termo_busca:
    cursor.execute("SELECT * FROM ordens WHERE cliente LIKE ?", ('%' + termo_busca + '%',))
else:
    cursor.execute("SELECT * FROM ordens")

dados = cursor.fetchall()

# ================= EXIBIÇÃO =================
if dados:
    # Converte para DataFrame automaticamente com nomes do banco
    colunas = [desc[0] for desc in cursor.description]
    df_os = pd.DataFrame(dados, columns=colunas)

    st.dataframe(df_os, use_container_width=True)

    # ================= EXPORTAR =================
    st.subheader("Exportar dados")

    def exportar_excel():
        output = io.BytesIO()
        with pd.ExcelWriter(output, engine='openpyxl') as writer:
            df_os.to_excel(writer, index=False)
        return output.getvalue()

    st.download_button(
        "📥 Baixar Excel",
        exportar_excel(),
        "ordens.xlsx"
    )

    # ================= FINALIZAR OS =================
    st.subheader("Finalizar OS")

    for row in dados:
        col1, col2, col3 = st.columns([4,2,2])

        with col1:
            st.write(f"**OS {row[0]}** - {row[2]} | {row[3]}")

        with col2:
            status = "🟢 Finalizado" if row[8] not in [None, ""] else "🟡 Pendente"
            st.write(status)

        with col3:
            if row[8] in [None, ""]:
                if st.button("Finalizar", key=row[0]):
                    finalizar_os(row[0])
                    st.rerun()

else:
    st.info("Nenhuma OS encontrada")
"""

with open("app.py", "w") as f:
      f.write(streamlit_code)
      # ================= NGROK =================
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError
import subprocess
import time

# Kill any ngrok processes already running to prevent 'endpoint already online' errors
# More aggressive kill command
subprocess.run(['killall', '-9', 'ngrok'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
ngrok.kill()

# Set your ngrok authtoken. You can get one from https://dashboard.ngrok.com/get-started/your-authtoken
# You can add it to Colab's secrets management (left panel, key icon) as 'NGROK_AUTH_TOKEN'
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN') # <-- Descomentado e configurado
ngrok.set_auth_token(NGROK_AUTH_TOKEN) # <-- Descomentado e configurado

# Start ngrok tunnel with retry logic
public_url = None
max_retries = 5
initial_delay = 5 # seconds

for i in range(max_retries):
    try:
        # Ensure any existing ngrok process is terminated first
        subprocess.run(['killall', '-9', 'ngrok'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        ngrok.kill()
        time.sleep(initial_delay) # Wait a bit after killing

        public_url = ngrok.connect(8501)
        print(f"Streamlit App URL: {public_url}")
        break # Exit loop if successful
    except PyngrokNgrokHTTPError as e:
        if "endpoint is already online" in str(e) or "ngrok client exception" in str(e):
            print(f"Encountered ngrok error: {e}. Attempt {i+1}/{max_retries}. Retrying in {initial_delay * (i+1)} seconds...")
            time.sleep(initial_delay * (i+1)) # Increasing sleep time
        else:
            raise e # Re-raise if it's a different type of ngrok error

if not public_url:
    raise Exception("Failed to establish ngrok tunnel after multiple retries.")

    # ================= RODAR =================
import subprocess
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

Streamlit App URL: NgrokTunnel: "https://facebook-enticing-aneurism.ngrok-free.dev" -> "http://localhost:8501"


<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

In [5]:
pip install reportlab streamlit pyngrok openpyxl